In [1]:
from pyspark.sql import functions as F

df_silver_teams = spark.sql("select * from silver_teams")
df_silver_matches = spark.sql("select * from silver_matches")
df_silver_players = spark.sql("select * from silver_players")

# Partidas como mandante
df_home = (df_silver_matches
    .groupBy(F.col("homeTeam").alias("id_team"))
    .agg(
        F.count("id").alias("matches_home")
    )
)

# Partidas como visitante
df_away = (df_silver_matches
    .groupBy(F.col("awayTeam").alias("id_team"))
    .agg(
        F.count("id").alias("matches_away")
    )
)

# Contagem de jogadores por time
df_squad_size = (df_silver_players
    .groupBy("id_team")
    .agg(F.count("id_player").alias("squad_size"))
)

df_gold_teams = (df_silver_teams
    .join(df_home, df_silver_teams.id == df_home.id_team, "left")
    .drop("id_team")
    .join(df_away, df_silver_teams.id == df_away.id_team, "left")
    .drop("id_team")
    .join(df_squad_size, df_silver_teams.id == df_squad_size.id_team, "left")
    .drop("id_team")
    .fillna(0, subset=["matches_home", "matches_away"])
    .withColumn("total_matches", F.col("matches_home") + F.col("matches_away"))
)

df_gold_teams.write.mode('overwrite').saveAsTable("gold_teams")
display(df_gold_teams)

StatementMeta(, 03a82b70-1548-4422-b6d0-72ab0fd1bb31, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 00ac959d-76c9-4fa4-9854-3ff224d72ebf)

In [2]:
df_gold_players = (df_silver_players
    .join(
        df_silver_teams.select(F.col("id").alias("id_team"), F.col("name").alias("team_name"), F.col("tla").alias("team_tla")),
        on="id_team",
        how="left"
    )
    .withColumn("age",
        F.floor(
            F.datediff(F.current_date(), F.to_date(F.col("dateOfBirth"), "yyyy-MM-dd")) / 365.25
        )
    )
    .select(
        "id_player",
        "name",
        "dateOfBirth",
        "age",
        "nationality",
        "position",
        "id_team",
        "team_name",
        "team_tla"
    )
)

df_gold_players.write.mode('overwrite').saveAsTable("gold_players")
display(df_gold_players)

StatementMeta(, 03a82b70-1548-4422-b6d0-72ab0fd1bb31, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bfbfcba7-0aa5-49f8-831c-6f4ef370446b)

In [3]:
df_gold_matches = (df_silver_matches
    .join(
        df_silver_teams.select(F.col("id").alias("homeTeam"), F.col("name").alias("homeTeam_name"), F.col("tla").alias("homeTeam_tla"), F.col("crest").alias("homeTeam_crest")),
        on="homeTeam",
        how="left"
    )
    .join(
        df_silver_teams.select(F.col("id").alias("awayTeam"), F.col("name").alias("awayTeam_name"), F.col("tla").alias("awayTeam_tla"), F.col("crest").alias("awayTeam_crest")),
        on="awayTeam",
        how="left"
    )
    .withColumn("match_date", F.to_date("utcDate"))
    .withColumn("match_time_utc", F.date_format("utcDate", "HH:mm"))
    .select(
        "id",
        "match_date",
        "match_time_utc",
        "stage",
        F.col("homeTeam").alias("homeTeam_id"),
        "homeTeam_name",
        "homeTeam_tla",
        "homeTeam_crest",
        F.col("awayTeam").alias("awayTeam_id"),
        "awayTeam_name",
        "awayTeam_tla",
        "awayTeam_crest"
    )
)

df_gold_matches.write.mode('overwrite').saveAsTable("gold_matches")
display(df_gold_matches)

StatementMeta(, 03a82b70-1548-4422-b6d0-72ab0fd1bb31, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c95096f8-a3d4-497a-82cb-dfca592b96bc)